# Lab Work - 6.8

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Q.1 Majority Voting

**01** Key difference: In classification each tree votes for a class. Final prediction is the **majority-vote class**:
$$ \hat{y} = \arg\max_k \sum_{i=1}^{n} \mathbb{I}(T_i(x) = k) $$

**02 Manual Example**

In [ ]:
votes = ['Malignant', 'Benign', 'Malignant', 'Malignant', 'Benign']
vote_count = Counter(votes)
print('Vote counts:', vote_count)

majority_class = max(vote_count, key=vote_count.get)
print('Predicted class (Majority vote):', majority_class)

prob_mal = vote_count['Malignant'] / len(votes)
print('Predicted probability (vote fraction for Malignant):', round(prob_mal, 3))

**03 Soft vs Hard Voting**

In [ ]:
# Tree probability outputs [P(Malignant), P(Benign)]
tree_probs = np.array([
    [0.8, 0.2],
    [0.3, 0.7],
    [0.9, 0.1],
    [0.75, 0.25],
    [0.4, 0.6]
])

soft_vote = np.mean(tree_probs, axis=0)
print('Soft-vote probabilities [Malignant, Benign]:', soft_vote)
print('Predicted class (soft):', 'Malignant' if soft_vote[0] > soft_vote[1] else 'Benign')
print('Confidence (max prob):', round(max(soft_vote), 3))

**04-06** (Theoretical)
- Soft voting is generally preferred because it uses the full probability information (confidence) from each tree.
- OOB accuracy: Each tree is trained on ~63% of data (bootstrap), ~37% are OOB and used as validation.
- OOB error converges as `n_estimators` increases (diminishing returns after ~200 trees).

## Q.2 Code It

In [ ]:
# 01 Load and split
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train distribution:', np.bincount(y_train))
print('Test distribution :', np.bincount(y_test))

In [ ]:
# 02 Baseline Random Forest
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf.fit(X_train, y_train)

print('Test Accuracy :', round(rf.score(X_test, y_test), 4))
print('OOB Accuracy  :', round(rf.oob_score_, 4))

In [ ]:
# 03 Classification Report + AUC
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Malignant (0)', 'Benign (1)']))

auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print('AUC-ROC:', round(auc, 4))

In [ ]:
# 04 Compare with single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

print('Decision Tree Test Accuracy:', round(dt.score(X_test, y_test), 4))
print('Random Forest Test Accuracy:', round(rf.score(X_test, y_test), 4))

print('DT AUC :', round(roc_auc_score(y_test, dt.predict_proba(X_test)[:, 1]), 4))
print('RF AUC :', round(auc, 4))

In [ ]:
# 05 Top 10 Feature Importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 6))
plt.title('Top 10 Feature Importances')
plt.barh(range(10), importances[indices][::-1])
plt.yticks(range(10), [feature_names[i] for i in indices[::-1]])
plt.xlabel('Importance')
plt.show()

In [ ]:
# 06 Soft-vote probabilities
probs = rf.predict_proba(X_test[:3])
print('Probabilities for first 3 test samples:\n', probs)
print('Sum per sample:', np.sum(probs, axis=1))

## Q.3 Hyperparameter Tuning

In [ ]:
# 01 Grid Search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'max_features': ['sqrt', 'log2', 0.5],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='f1_weighted', n_jobs=-1
)
grid.fit(X_train, y_train)

print('Best parameters:', grid.best_params_)
best_rf = grid.best_estimator_

In [ ]:
# 02 Best model performance
y_pred_best = best_rf.predict(X_test)
print('Best RF Test Accuracy :', round(accuracy_score(y_test, y_pred_best), 4))
print('Best RF Weighted F1   :', round(classification_report(y_test, y_pred_best, output_dict=True)['weighted avg']['f1-score'], 4))
print('Best RF AUC-ROC       :', round(roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1]), 4))

In [ ]:
# 03-06 Visualizations (Feature Importance, ROC, Confusion Matrix)
# Feature importance with std
importances = best_rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in best_rf.estimators_], axis=0)
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(12, 8))
plt.title('Feature Importances (Top 15)')
plt.barh(range(15), importances[indices], xerr=std[indices], align='center')
plt.yticks(range(15), [feature_names[i] for i in indices])
plt.xlabel('Mean Importance ± Std')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
y_prob = best_rf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'RF (AUC = {roc_auc_score(y_test, y_prob):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (One-vs-Rest)')
plt.legend()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Malignant', 'Benign'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## Q.4 Deep Intuition

**01** Perfect train accuracy with lower OOB/test accuracy is normal — trees overfit individual bootstrap samples. OOB provides an unbiased estimate of generalization performance.

**02** Comparison table (bias, variance, interpretability, scaling) can be filled here.

**03** `max_features='sqrt'` reduces correlation between trees → lowers variance of the ensemble (bagging + feature randomness).

**04** For fraud detection (high recall target), use `predict_proba`, plot ROC/PR curve, choose threshold that achieves ≥90% Recall while monitoring Precision drop.